## Google Reviews - Raw to Bronze Processing

In [0]:
from pyspark.sql.functions import *
import uuid

SOURCE_VOLUME = "/Volumes/fuel_project_dev/raw/fuel_transactions"
CHECKPOINT_LOCATION = "/Volumes/fuel_project_dev/checkpoints/bronze/fuel_transactions"
TARGET_TABLE = "fuel_project_dev.bronze.fuel_transactions"


In [0]:
load_id = str(uuid.uuid4())

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","json")
    .option("cloudFiles.schemaLocation", CHECKPOINT_LOCATION)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.maxFilesPerTrigger", 100)
    .load(SOURCE_VOLUME)
    )

In [0]:
df = df.withColumn("_event_date", to_date(to_timestamp(col("StartTime"), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS")))\
    .withColumn("_bronze_ingestion_timestamp", current_timestamp())\
    .withColumn("_bronze_source_file_path", col("_metadata.file_path"))\
    .withColumn("load_id", lit(load_id))

In [0]:
query = (
    df.writeStream
    .trigger(availableNow=True)
    .format("delta")
    .outputMode("append")
    .option("mergeSchema", "true")
    .option("checkpointLocation",CHECKPOINT_LOCATION)
    .table(TARGET_TABLE)
)

In [0]:
# If there is a print statement after streaming query, then it immediately stops the execution
# Streaming query is stopped immediately without this statement in Jobs
query.awaitTermination()